In [ ]:
import csv
import json
import time
import pandas as pd
import requests
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import numpy as np

# ── CONFIG ────────────────────────────────────────────────────────────────────
API_KEY          = "7NWMGEG2B88AC3ZQQ7NFBZ7TQVAQI6TC3P"
ETHERSCAN_URL    = "https://api.etherscan.io/v2/api"
CHAIN_ID         = 1
PIPELINE_VERSION = "5.0"

BASE_DIR = Path("/kaggle/working/dataset")

# ══════════════════════════════════════════════════════════════════════════════
TARGET_LABEL = 0   # ← ONLY LINE YOU NEED TO CHANGE BETWEEN RUNS
# ══════════════════════════════════════════════════════════════════════════════

# ── CLASS MAPPING ─────────────────────────────────────────────────────────────
CLASS_MAP = {
    0: "benign",
    1: "phishing",
    2: "rug_pull",
    3: "ponzi",
    4: "flash_loan_attack",
    5: "malicious_mev",
    6: "exploit_contract",
    7: "honeypot",
}

PER_CLASS_CAP = {
    "benign":             3548,
    "phishing":            462,
    "rug_pull":          10000,
    "ponzi":            10000,
    "flash_loan_attack": 20000,
    "malicious_mev":     10000,
    "exploit_contract":  10000,
    "honeypot":            101,
}

# ── AUTO-DERIVED FROM TARGET_LABEL ────────────────────────────────────────────
TARGET_CLASS = CLASS_MAP[TARGET_LABEL]
CAP          = PER_CLASS_CAP[TARGET_CLASS]

print(f"▶ Running label={TARGET_LABEL}  class='{TARGET_CLASS}'  cap={CAP:,}")

# ── DIRECTORIES ───────────────────────────────────────────────────────────────
class_dir = BASE_DIR / TARGET_CLASS
for sub in [
    class_dir / "transactions",
    class_dir / "token" / "erc-20",
    class_dir / "token" / "erc-721",
    class_dir / "token" / "erc-1155",
    class_dir / "address",
    class_dir / "internal_transactions",
    class_dir / "failed_transactions",
]:
    sub.mkdir(exist_ok=True, parents=True)

# ── LABEL-SPECIFIC OUTPUT FILES ───────────────────────────────────────────────
CHECKPOINT_FILE = BASE_DIR / f"checkpoint_label{TARGET_LABEL}.txt"
MASTER_CSV      = BASE_DIR / f"master_dataset_label{TARGET_LABEL}.csv"
FAILED_LOG      = BASE_DIR / f"failed_addresses_label{TARGET_LABEL}.csv"
OUTLIER_LOG     = BASE_DIR / f"outlier_addresses_label{TARGET_LABEL}.csv"

MAX_TX_PER_PAGE = 10000
SLEEP           = 0.4

# ── INIT LOG FILES ────────────────────────────────────────────────────────────
if not FAILED_LOG.exists():
    with open(FAILED_LOG, 'w', newline='') as f:
        csv.writer(f, quoting=csv.QUOTE_ALL).writerow(
            ["address", "label", "class_name", "error", "timestamp"]
        )

if not OUTLIER_LOG.exists():
    with open(OUTLIER_LOG, 'w', newline='') as f:
        csv.writer(f, quoting=csv.QUOTE_ALL).writerow(
            ["address", "label", "class_name",
             "tx_count", "cap_applied", "timestamp"]
        )

# ── HELPERS ───────────────────────────────────────────────────────────────────
def get_class_dirs(label):
    class_name = CLASS_MAP.get(int(label))
    if class_name is None:
        raise ValueError(f"Unknown label '{label}'.")
    cd = BASE_DIR / class_name
    return {
        "class_dir":    cd,
        "trans_dir":    cd / "transactions",
        "internal_dir": cd / "internal_transactions",
        "failed_dir":   cd / "failed_transactions",
        "erc20_dir":    cd / "token" / "erc-20",
        "erc721_dir":   cd / "token" / "erc-721",
        "erc1155_dir":  cd / "token" / "erc-1155",
        "addr_dir":     cd / "address",
        "class_name":   class_name,
    }


def fetch_etherscan(address, action, page=1, offset=10000,
                    startblock=0, endblock=99999999):
    params = {
        "chainid": CHAIN_ID, "module": "account", "action": action,
        "address": address, "startblock": startblock, "endblock": endblock,
        "page": page, "offset": offset, "sort": "asc", "apikey": API_KEY,
    }
    for attempt in range(3):
        try:
            r = requests.get(ETHERSCAN_URL, params=params, timeout=15)
            d = r.json()
            if d['status'] == '1':
                return {"status": "1", "message": d.get("message", "OK"),
                        "result": d['result']}
            elif 'rate limit' in str(d.get('result', '')).lower():
                print("    ⏳ Rate limit — sleeping 10 s")
                time.sleep(10)
                continue
            else:
                return {"status": "0", "message": d.get("message", "No data"),
                        "result": [] if action != "balance" else "0"}
        except Exception as e:
            print(f"    ❌ Attempt {attempt + 1} failed: {e}")
            time.sleep(2)
    return {"status": "0", "message": "Error after retries",
            "result": [] if action != "balance" else "0"}


def is_contract(address):
    params = {
        "chainid": CHAIN_ID, "module": "proxy", "action": "eth_getCode",
        "address": address, "tag": "latest", "apikey": API_KEY,
    }
    try:
        r    = requests.get(ETHERSCAN_URL, params=params, timeout=10)
        code = r.json().get('result', '0x')
        size = (len(code) - 2) // 2 if len(code) > 2 else 0
        return size > 0, size
    except Exception:
        return False, 0


def fetch_all_paginated(address, action, cap: int):
    all_results = []
    startblock  = 0
    endblock    = 99999999
    truncated   = False

    while True:
        data = fetch_etherscan(address, action,
                               page=1, offset=MAX_TX_PER_PAGE,
                               startblock=startblock, endblock=endblock)

        if data['status'] == '0' or not data['result']:
            break

        batch     = data['result']
        remaining = cap - len(all_results)

        if len(batch) >= remaining:
            all_results.extend(batch[:remaining])
            truncated = True
            print(f"    ⚠️  Cap hit at {cap:,} txs [{action}] — stopping early")
            break

        all_results.extend(batch)

        if len(batch) < MAX_TX_PER_PAGE:
            break

        last_block = int(batch[-1]['blockNumber'])
        if last_block >= endblock:
            break

        startblock = last_block + 1
        print(f"    ↳ {action}: next startblock={startblock} "
              f"(total so far: {len(all_results):,})")
        time.sleep(0.2)

    return {
        "status":        "1" if all_results else "0",
        "message":       "OK" if all_results else "No data found",
        "result":        all_results,
        "truncated":     truncated,
        "total_fetched": len(all_results),
    }


def save_checkpoint(address, stats_row):
    with open(CHECKPOINT_FILE, 'a') as f:
        f.write(f"{address}\n")
    df_row = pd.DataFrame([stats_row])
    if not MASTER_CSV.exists():
        df_row.to_csv(MASTER_CSV, index=False)
    else:
        df_row.to_csv(MASTER_CSV, mode='a', header=False, index=False)


def log_failed(address, label, class_name, error):
    with open(FAILED_LOG, 'a', newline='') as f:
        csv.writer(f, quoting=csv.QUOTE_ALL).writerow([
            address, label, class_name,
            str(error), datetime.now(timezone.utc).isoformat()
        ])


def log_outlier(address, label, class_name, tx_count, cap):
    with open(OUTLIER_LOG, 'a', newline='') as f:
        csv.writer(f, quoting=csv.QUOTE_ALL).writerow([
            address, label, class_name,
            tx_count, cap,
            datetime.now(timezone.utc).isoformat()
        ])


# ── LOAD + FILTER TO TARGET LABEL ONLY ───────────────────────────────────────
api_df = pd.read_csv("/kaggle/input/datasets/kasem2026/final-multi-class-etherium-attack-dataset/master_dataset_final_8class.csv")
api_df = api_df[api_df['label'] == TARGET_LABEL].reset_index(drop=True)
print(f"Addresses for label={TARGET_LABEL} ({TARGET_CLASS}): {len(api_df):,}")

# ── RESUME FROM CHECKPOINT ────────────────────────────────────────────────────
if CHECKPOINT_FILE.exists():
    completed = set(CHECKPOINT_FILE.read_text().strip().split('\n'))
    completed.discard('')
    api_df = api_df[~api_df['address'].isin(completed)].reset_index(drop=True)
    print(f"Already done: {len(completed):,}  |  Remaining: {len(api_df):,}")

# ── MAIN LOOP ─────────────────────────────────────────────────────────────────
outlier_count_session = 0
failed_count_session  = 0

for idx, row in api_df.iterrows():
    addr  = row['address']
    label = TARGET_LABEL

    print(f"\n[{idx + 1}/{len(api_df)}] {addr}  (label={label} / {TARGET_CLASS})")

    try:
        dirs = get_class_dirs(label)

        collection_timestamp = datetime.now(timezone.utc).isoformat()
        now_ts               = datetime.now(timezone.utc).timestamp()

        # ── Quick outlier pre-check ───────────────────────────────────────────
        pre_check = fetch_etherscan(addr, "txlist",
                                    page=1, offset=MAX_TX_PER_PAGE)
        pre_count = len(pre_check.get('result', []))

        if pre_count == MAX_TX_PER_PAGE and CAP <= MAX_TX_PER_PAGE:
            print(f"  🚫 Outlier — first page full ({pre_count:,} txs) "
                  f"exceeds cap ({CAP:,}). Logging and skipping.")
            log_outlier(addr, label, TARGET_CLASS, f">{pre_count}", CAP)
            outlier_count_session += 1
            with open(CHECKPOINT_FILE, 'a') as f:
                f.write(f"{addr}\n")
            continue

        # ── Fetch all data (capped) ───────────────────────────────────────────
        is_contract_addr, bytecode_size = is_contract(addr)

        normal_tx   = fetch_all_paginated(addr, "txlist",         cap=CAP)
        internal_tx = fetch_all_paginated(addr, "txlistinternal", cap=CAP)
        erc20_tx    = fetch_all_paginated(addr, "tokentx",        cap=CAP)
        erc721_tx   = fetch_all_paginated(addr, "tokennfttx",     cap=CAP)
        erc1155_tx  = fetch_all_paginated(addr, "token1155tx",    cap=CAP)
        balance     = fetch_etherscan(addr, "balance", 1, 1)

        raw_bal     = balance.get('result', '0')
        balance_wei = int(raw_bal) if str(raw_bal).isdigit() else 0

        successful_tx        = [tx for tx in normal_tx['result'] if tx.get('isError') == '0']
        failed_tx            = [tx for tx in normal_tx['result'] if tx.get('isError') == '1']
        contract_creation_tx = [tx for tx in normal_tx['result'] if tx.get('contractAddress')]
        approval_tx          = [tx for tx in normal_tx['result']
                                if tx.get('input', '')[:10] == '0x095ea7b3']

        # ── Stats dict ────────────────────────────────────────────────────────
        stats = {
            'address':             addr,
            'label':               label,
            'class_name':          TARGET_CLASS,
            'collected_at_utc':    collection_timestamp,
            'pipeline_version':    PIPELINE_VERSION,
            'is_contract':         is_contract_addr,
            'bytecode_size':       bytecode_size,
            'balance_wei':         balance_wei,
            'balance_eth':         balance_wei / 1e18,
            'total_tx_count':      len(normal_tx['result']),
            'successful_tx_count': len(successful_tx),
            'failed_tx_count':     len(failed_tx),
            'internal_tx_count':   len(internal_tx['result']),
            'erc20_tx_count':      len(erc20_tx['result']),
            'erc721_tx_count':     len(erc721_tx['result']),
            'erc1155_tx_count':    len(erc1155_tx['result']),
            'contract_creation_count': len(contract_creation_tx),
            'approval_count':      len(approval_tx),
            'cap_applied':         CAP,
            'data_is_partial':     any([
                                       normal_tx['truncated'],
                                       internal_tx['truncated'],
                                       erc20_tx['truncated'],
                                   ]),
            'normal_tx_truncated':   normal_tx['truncated'],
            'internal_tx_truncated': internal_tx['truncated'],
            'erc20_tx_truncated':    erc20_tx['truncated'],
            # Working sets (removed before save)
            'unique_receivers':          set(),
            'unique_senders':            set(),
            'total_value_sent_wei':      0,
            'total_value_received_wei':  0,
            'total_gas_used':            0,
            'avg_gas_price':             0,
            'first_tx_timestamp':        None,
            'last_tx_timestamp':         None,
            'unique_erc20_tokens':       set(),
            'unique_erc721_tokens':      set(),
            'unique_erc1155_tokens':     set(),
            'method_ids':                Counter(),
            'unique_method_ids':         set(),
            'unique_approved_contracts': set(),
        }

        gas_prices        = []
        tx_timestamps     = []
        zero_value_count  = 0
        first_approval_ts = None
        last_approval_ts  = None

        for tx in normal_tx['result']:
            tx_from = (tx.get('from') or '').lower()
            tx_to   = (tx.get('to')   or '').lower()
            value   = int(tx.get('value', 0))

            if tx_from == addr.lower():
                if tx_to:
                    stats['unique_receivers'].add(tx_to)
                stats['total_value_sent_wei'] += value
            else:
                if tx_from:
                    stats['unique_senders'].add(tx_from)
                stats['total_value_received_wei'] += value

            if value == 0:
                zero_value_count += 1

            stats['total_gas_used'] += int(tx.get('gasUsed', 0))
            if tx.get('gasPrice'):
                gas_prices.append(int(tx['gasPrice']))

            input_data = tx.get('input', '0x')
            if len(input_data) >= 10:
                method_id = input_data[:10]
                stats['method_ids'][method_id] += 1
                stats['unique_method_ids'].add(method_id)
                if method_id == '0x095ea7b3' and tx_to:
                    stats['unique_approved_contracts'].add(tx_to)

            timestamp = int(tx.get('timeStamp', 0))
            tx_timestamps.append(timestamp)

            if stats['first_tx_timestamp'] is None:
                stats['first_tx_timestamp'] = timestamp
            stats['last_tx_timestamp'] = timestamp

            if input_data[:10] == '0x095ea7b3':
                if first_approval_ts is None:
                    first_approval_ts = timestamp
                last_approval_ts = timestamp

        stats['avg_gas_price']       = sum(gas_prices) / len(gas_prices) if gas_prices else 0
        stats['zero_value_tx_count'] = zero_value_count

        if len(tx_timestamps) >= 2:
            deltas = [tx_timestamps[i+1] - tx_timestamps[i]
                      for i in range(len(tx_timestamps) - 1)]
            stats['avg_time_between_tx_sec'] = float(np.mean(deltas))
            stats['std_time_between_tx_sec'] = float(np.std(deltas))
        else:
            stats['avg_time_between_tx_sec'] = None
            stats['std_time_between_tx_sec'] = None

        stats['first_approval_timestamp'] = first_approval_ts
        stats['last_approval_timestamp']  = last_approval_ts

        for tx in erc20_tx['result']:
            ca = (tx.get('contractAddress') or '').lower()
            if ca:
                stats['unique_erc20_tokens'].add(ca)

        for tx in erc721_tx['result']:
            ca = (tx.get('contractAddress') or '').lower()
            if ca:
                stats['unique_erc721_tokens'].add(ca)

        for tx in erc1155_tx['result']:
            ca = (tx.get('contractAddress') or '').lower()
            if ca:
                stats['unique_erc1155_tokens'].add(ca)

        stats['first_tx_datetime'] = (
            pd.to_datetime(stats['first_tx_timestamp'], unit='s')
            if stats['first_tx_timestamp'] is not None else None
        )
        stats['last_tx_datetime'] = (
            pd.to_datetime(stats['last_tx_timestamp'], unit='s')
            if stats['last_tx_timestamp'] is not None else None
        )
        stats['active_span_days'] = (
            (stats['last_tx_timestamp'] - stats['first_tx_timestamp']) / 86400
            if (stats['first_tx_timestamp'] and stats['last_tx_timestamp'])
            else 0
        )
        stats['account_age_days'] = (
            (now_ts - stats['first_tx_timestamp']) / 86400
            if stats['first_tx_timestamp'] is not None else 0
        )

        # ── Scalar counts ─────────────────────────────────────────────────────
        stats['unique_receivers_count']          = len(stats['unique_receivers'])
        stats['unique_senders_count']            = len(stats['unique_senders'])
        stats['unique_erc20_tokens_count']       = len(stats['unique_erc20_tokens'])
        stats['unique_erc721_tokens_count']      = len(stats['unique_erc721_tokens'])
        stats['unique_erc1155_tokens_count']     = len(stats['unique_erc1155_tokens'])
        stats['unique_method_ids_count']         = len(stats['unique_method_ids'])
        stats['unique_approved_contracts_count'] = len(stats['unique_approved_contracts'])

        if stats['method_ids']:
            stats['most_common_method_id']    = stats['method_ids'].most_common(1)[0][0]
            stats['most_common_method_count'] = stats['method_ids'].most_common(1)[0][1]
        else:
            stats['most_common_method_id']    = None
            stats['most_common_method_count'] = 0

        total = stats['total_tx_count']
        stats['failed_tx_ratio'] = stats['failed_tx_count'] / total if total else 0
        stats['sent_to_received_value_ratio'] = (
            stats['total_value_sent_wei'] / stats['total_value_received_wei']
            if stats['total_value_received_wei'] else None
        )
        outgoing = sum(
            1 for tx in normal_tx['result']
            if (tx.get('from') or '').lower() == addr.lower()
        )
        stats['fan_out_ratio'] = (
            stats['unique_receivers_count'] / outgoing if outgoing else 0
        )

        # ── ETH convenience columns ───────────────────────────────────────────
        stats['total_value_sent_eth']     = stats['total_value_sent_wei']     / 1e18
        stats['total_value_received_eth'] = stats['total_value_received_wei'] / 1e18
        stats['avg_gas_price_gwei']       = stats['avg_gas_price'] / 1e9

        # ── Remove working sets before CSV write ──────────────────────────────
        for key in ['unique_receivers', 'unique_senders', 'unique_erc20_tokens',
                    'unique_erc721_tokens', 'unique_erc1155_tokens',
                    'method_ids', 'unique_method_ids', 'unique_approved_contracts']:
            del stats[key]

        # ── Print summary ─────────────────────────────────────────────────────
        partial_flag = "⚠️  PARTIAL" if stats['data_is_partial'] else "✅ FULL"
        print(f"  txs={stats['total_tx_count']:,}  bal={stats['balance_eth']:.4f} ETH  {partial_flag}")

        # ── Save JSON files ───────────────────────────────────────────────────
        if normal_tx['result']:
            with open(dirs['trans_dir'] / f"{addr}.json", 'w') as f:
                json.dump(normal_tx, f, indent=2)

        if internal_tx['result']:
            with open(dirs['internal_dir'] / f"{addr}.json", 'w') as f:
                json.dump(internal_tx, f, indent=2)

        if failed_tx:
            with open(dirs['failed_dir'] / f"{addr}.json", 'w') as f:
                json.dump({"status": "1", "result": failed_tx}, f, indent=2)

        for dir_key, data in [('erc20_dir',   erc20_tx),
                               ('erc721_dir',  erc721_tx),
                               ('erc1155_dir', erc1155_tx)]:
            with open(dirs[dir_key] / f"{addr}.json", 'w') as f:
                json.dump({
                    "status":    "1" if data['result'] else "0",
                    "message":   "OK" if data['result'] else "No token found",
                    "result":    data['result'],
                    "truncated": data['truncated'],
                }, f, indent=2)

        with open(dirs['addr_dir'] / f"{addr}.json", 'w') as f:
            json.dump({
                "collected_at_utc":    collection_timestamp,
                "pipeline_version":    PIPELINE_VERSION,
                "class_name":          TARGET_CLASS,
                "cap_applied":         CAP,
                "data_is_partial":     stats['data_is_partial'],
                "truncation_flags": {
                    "normal_tx":   normal_tx['truncated'],
                    "internal_tx": internal_tx['truncated'],
                    "erc20_tx":    erc20_tx['truncated'],
                    "erc721_tx":   erc721_tx['truncated'],
                    "erc1155_tx":  erc1155_tx['truncated'],
                },
                "statuses": {
                    "normal_tx":   normal_tx['status'],
                    "internal_tx": internal_tx['status'],
                    "erc20_tx":    erc20_tx['status'],
                    "erc721_tx":   erc721_tx['status'],
                    "erc1155_tx":  erc1155_tx['status'],
                    "balance":     balance['status'],
                },
                "balance_wei":             balance_wei,
                "total_tx_count":          stats['total_tx_count'],
                "internal_tx_count":       stats['internal_tx_count'],
                "erc20_tx_count":          stats['erc20_tx_count'],
                "erc721_tx_count":         stats['erc721_tx_count'],
                "erc1155_tx_count":        stats['erc1155_tx_count'],
                "failed_tx_count":         stats['failed_tx_count'],
                "contract_creation_count": stats['contract_creation_count'],
                "approval_count":          stats['approval_count'],
            }, f, indent=2)

        save_checkpoint(addr, stats)
        print("  ✅ Checkpointed")

    except Exception as e:
        print(f"  ❌ FAILED: {e}")
        log_failed(addr, label, TARGET_CLASS, e)
        failed_count_session += 1
        continue

    time.sleep(SLEEP)

# ── SESSION SUMMARY ───────────────────────────────────────────────────────────
print(f"\n{'═'*60}")
print(f"  Run complete  [label={TARGET_LABEL} / {TARGET_CLASS}]")
print(f"  Outliers skipped  : {outlier_count_session:,}")
print(f"  Failed            : {failed_count_session:,}")
print(f"  Master dataset    : {MASTER_CSV}")
print(f"  Outlier log       : {OUTLIER_LOG}")
print(f"  Failed log        : {FAILED_LOG}")
print(f"{'═'*60}")